# Find common TikZ/LaTeX n-grams

Standalone notebook.

Pipeline:

```text
Parquet files
→ read TikZ/LaTeX code
→ remove comments
→ normalize numbers to NUM
→ tokenize
→ count global 1-, 2-, 3-, and 4-grams
→ select TOP_K most frequent n-grams
→ save them as filter
```


In [18]:
from collections import Counter
from pathlib import Path
import json
import re

import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm


## Configuration

In [19]:
PARQUET_DIR = Path("/home/jonas/Datasets/TikZ/DatikZ-v4-raw")
OUTPUT_DIR = Path("./repetition_test")

CODE_COLUMNS = (
    "reference_code",
    "code",
    "tikz_code",
)

NGRAM_ORDERS = (20, 50)

TOP_K = 200
BATCH_SIZE = 2048

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FILTER_FILE = OUTPUT_DIR / "common_ngrams.json"
CSV_FILE = OUTPUT_DIR / "common_ngrams.csv"


## Normalize and tokenize

In [20]:
NUMBER_RE = re.compile(
    r"""
    (?<![A-Za-z])
    [-+]?
    (?:
        \d+\.\d*
        |
        \.\d+
        |
        \d+
    )
    (?:[eE][-+]?\d+)?
    """,
    re.VERBOSE,
)

TOKEN_RE = re.compile(
    r"""
    \\[A-Za-z@]+
    | NUM
    | [A-Za-z_][A-Za-z0-9_]*
    | --|->|<-|<->
    | [^\s]
    """,
    re.VERBOSE,
)


def strip_comments(code: str) -> str:
    return "\n".join(
        re.sub(
            r"(?<!\\)%.*$",
            "",
            line,
        )
        for line in str(code or "").splitlines()
    )


def normalize(code: str) -> str:
    code = strip_comments(code)

    return NUMBER_RE.sub(
        "NUM",
        code,
    )


def tokenize(code: str) -> list[str]:
    return TOKEN_RE.findall(
        normalize(code)
    )


## n-gram helpers

In [21]:
def make_ngrams(
    tokens: list[str],
    n: int,
):
    for i in range(
        len(tokens) - n + 1
    ):
        yield tuple(
            tokens[i:i + n]
        )


def find_code_column(
    parquet_file: Path,
) -> str:
    columns = set(
        pq.ParquetFile(
            parquet_file
        ).schema.names
    )

    for column in CODE_COLUMNS:
        if column in columns:
            return column

    raise ValueError(
        f"No code column found in {parquet_file}. "
        f"Tried: {CODE_COLUMNS}"
    )


## Count global n-grams

In [ ]:
parquet_files = sorted(
    PARQUET_DIR.rglob("*.parquet")
)

if not parquet_files:
    raise FileNotFoundError(
        f"No Parquet files found in "
        f"{PARQUET_DIR}"
    )

counts = Counter()
samples = 0

for parquet_file in tqdm(
    parquet_files,
    desc="Parquet files",
):
    code_column = find_code_column(
        parquet_file
    )

    parquet = pq.ParquetFile(
        parquet_file
    )

    for batch in parquet.iter_batches(
        columns=[code_column],
        batch_size=BATCH_SIZE,
    ):
        for code in (
            batch
            .column(0)
            .to_pylist()
        ):
            tokens = tokenize(
                str(code or "")
            )

            for n in NGRAM_ORDERS:
                counts.update(
                    (n, gram)
                    for gram in make_ngrams(
                        tokens,
                        n,
                    )
                )

            samples += 1

print(
    f"Samples: {samples:,}"
)

print(
    f"Unique n-grams: {len(counts):,}"
)


Parquet files:  28%|██▊       | 24/86 [01:58<04:54,  4.75s/it]

## Top-K most frequent n-grams

In [ ]:
most_common = counts.most_common(
    TOP_K
)

rows = []

for rank, (
    (n, gram),
    count,
) in enumerate(
    most_common,
    start=1,
):
    rows.append({
        "rank": rank,
        "n": n,
        "count": count,
        "ngram": " ".join(gram),
        "tokens": list(gram),
    })

top_ngrams = pd.DataFrame(
    rows
)

display(
    top_ngrams
)


,rank,n,count,ngram,tokens
0,1,1,33365394,NUM,[NUM]
1,2,1,18233974,",","[,]"
2,3,1,12306672,),[)]
3,4,1,12304461,(,[(]
4,5,1,12160152,{,[{]
...,...,...,...,...,...
195,196,1,427823,\documentclass,[\documentclass]
196,197,3,427823,\begin { document,"[\begin, {, document]"
197,198,4,427823,\begin { document },"[\begin, {, document, }]"
198,199,3,427705,\end { document,"[\end, {, document]"


## Save filter

In [ ]:
filter_data = [
    {
        "rank": row["rank"],
        "n": row["n"],
        "count": row["count"],
        "tokens": row["tokens"],
    }
    for row in rows
]

FILTER_FILE.write_text(
    json.dumps(
        filter_data,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

top_ngrams.drop(
    columns=["tokens"]
).to_csv(
    CSV_FILE,
    index=False,
)

print(
    "Saved:",
    FILTER_FILE,
)

print(
    "Saved:",
    CSV_FILE,
)


Saved: repetition_test/common_ngrams.json
Saved: repetition_test/common_ngrams.csv


## Build filter set

In [ ]:
common_ngrams = {
    (
        item["n"],
        tuple(item["tokens"]),
    )
    for item in filter_data
}

print(
    f"Filter size: "
    f"{len(common_ngrams):,}"
)


Filter size: 200


## Test how much of a sample is covered by the filter

In [ ]:
def filter_coverage(
    code: str,
) -> dict:
    tokens = tokenize(code)

    total = 0
    filtered = 0

    by_n = {}

    for n in NGRAM_ORDERS:
        grams = list(
            make_ngrams(
                tokens,
                n,
            )
        )

        n_total = len(grams)

        n_filtered = sum(
            (n, gram) in common_ngrams
            for gram in grams
        )

        by_n[n] = {
            "total": n_total,
            "filtered": n_filtered,
            "coverage": (
                n_filtered / n_total
                if n_total
                else 0.0
            ),
        }

        total += n_total
        filtered += n_filtered

    return {
        "total_ngrams": total,
        "filtered_ngrams": filtered,
        "coverage": (
            filtered / total
            if total
            else 0.0
        ),
        "by_n": by_n,
    }


In [ ]:
example = r"""
\begin{tikzpicture}
    \draw (0,0) -- (1,1);
    \draw (1,1) -- (2,2);
    \node at (3,4) {Example};
\end{tikzpicture}
"""

print(normalize(example))
print()
print(filter_coverage(example))



\begin{tikzpicture}
    \draw (NUM,NUM) -- (NUM,NUM);
    \draw (NUM,NUM) -- (NUM,NUM);
    \node at (NUM,NUM) {Example};
\end{tikzpicture}

{'total_ngrams': 174, 'filtered_ngrams': 137, 'coverage': 0.7873563218390804, 'by_n': {1: {'total': 45, 'filtered': 44, 'coverage': 0.9777777777777777}, 2: {'total': 44, 'filtered': 39, 'coverage': 0.8863636363636364}, 3: {'total': 43, 'filtered': 31, 'coverage': 0.7209302325581395}, 4: {'total': 42, 'filtered': 23, 'coverage': 0.5476190476190477}}}


## Optional: show Top-K separately per n-gram order

This is useful if the global Top-K is dominated by 1-grams.


In [ ]:
TOP_K_PER_ORDER = 200
OUTPUT_FILE = "top_ngrams.csv"

all_results = []

for n in NGRAM_ORDERS:
    part = [
        (
            gram,
            count,
        )
        for (
            gram_n,
            gram
        ), count in counts.items()
        if gram_n == n
    ]

    part.sort(
        key=lambda x: x[1],
        reverse=True,
    )

    df = pd.DataFrame(
        [
            {
                "rank": rank,
                "n": n,
                "count": count,
                "ngram": " ".join(gram),
            }
            for rank, (
                gram,
                count,
            ) in enumerate(
                part[:TOP_K_PER_ORDER],
                start=1,
            )
        ]
    )

    display(df)
    all_results.append(df)


pd.concat(
    all_results,
    ignore_index=True,
).to_csv(
    OUTPUT_FILE,
    index=False,
)

print(f"Gespeichert: {OUTPUT_FILE}")

,rank,n,count,ngram
0,1,1,33365394,NUM
1,2,1,18233974,","
2,3,1,12306672,)
3,4,1,12304461,(
4,5,1,12160152,{
...,...,...,...,...
195,196,1,54524,F
196,197,1,54091,outer
197,198,1,51875,error
198,199,1,51333,xlabel


,rank,n,count,ngram
0,1,2,9275961,"NUM ,"
1,2,2,9057691,", NUM"
2,3,2,8268035,NUM )
3,4,2,8059192,( NUM
4,5,2,7797456,NUM NUM
...,...,...,...,...
195,196,2,134402,", calc"
196,197,2,134294,line join
197,198,2,134188,{ tikzcd
198,199,2,133359,tikzcd }


,rank,n,count,ngram
0,1,3,8122196,"NUM , NUM"
1,2,3,7214974,", NUM )"
2,3,3,6643640,"( NUM ,"
3,4,3,6411015,NUM NUM NUM
4,5,3,2456682,) -- (
...,...,...,...,...
195,196,3,124982,) $ )
196,197,3,123747,) -- +
197,198,3,121187,NUM ) [
198,199,3,120637,{ xcolor }


,rank,n,count,ngram
0,1,4,7048410,"NUM , NUM )"
1,2,4,6582417,"( NUM , NUM"
2,3,4,6227468,NUM NUM NUM NUM
3,4,4,2074963,", NUM ) --"
4,5,4,1977198,NUM ) -- (
...,...,...,...,...
195,196,4,104755,"decorations . pathreplacing ,"
196,197,4,104437,circle ( NUM )
197,198,4,104310,", decorations . pathreplacing"
198,199,4,103311,fill opacity = NUM


Gespeichert: top_ngrams.csv
